<a href="https://colab.research.google.com/github/jayasirivasamsetti-glitch/jayasireesha/blob/main/Screen_rotation_has_orthogonal_transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Make numerical output easier to read
np.set_printoptions(precision=4, suppress=True)

print("Libraries imported successfully.")

In [ ]:
# ============================================================
# USER CONFIGURATION / INPUTS
# ============================================================

# Phone screen dimensions in pixels
SCREEN_WIDTH = 1080
SCREEN_HEIGHT = 1920

# Rotation angle
# 90 degrees = portrait -> landscape geometric rotation
ROTATION_ANGLE = 90

# Screen coordinates of a point
# Coordinates are measured from the CENTER of the screen.
POINT = np.array([300.0, 500.0])

# A second point for distance-preservation testing
POINT_2 = np.array([-200.0, 400.0])

print("Screen width  :", SCREEN_WIDTH)
print("Screen height :", SCREEN_HEIGHT)
print("Rotation angle:", ROTATION_ANGLE, "degrees")
print("Point 1       :", POINT)
print("Point 2       :", POINT_2)

In [ ]:
# ============================================================
# MATHEMATICAL MODEL
# ============================================================

print("""
For a 2D rotation through angle θ:

              | cosθ   -sinθ |
      Q   =   |              |
              | sinθ    cosθ |

A point P = [x, y]^T is transformed as:

      P' = QP

For an orthogonal matrix:

      Q^T Q = I

Therefore:

      ||QP||²
      = (QP)^T(QP)
      = P^T Q^T QP
      = P^T P

Hence distances and lengths are preserved.
""")

In [ ]:
# ============================================================
# ROTATION MATRIX
# ============================================================

# Convert degrees to radians
theta = np.deg2rad(ROTATION_ANGLE)

# Standard 2D counterclockwise rotation matrix
Q = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

print("Rotation angle in radians:", theta)

print("\nRotation matrix Q:")
print(Q)

In [ ]:
# ============================================================
# SYMBOLIC VERIFICATION USING SYMPY
# ============================================================

theta_symbol = sp.symbols('theta', real=True)

Q_symbol = sp.Matrix([
    [sp.cos(theta_symbol), -sp.sin(theta_symbol)],
    [sp.sin(theta_symbol),  sp.cos(theta_symbol)]
])

print("Symbolic rotation matrix Q:")
sp.pprint(Q_symbol)

# Calculate Q^T Q
QTQ_symbol = sp.simplify(Q_symbol.T * Q_symbol)

print("\nQ^T Q:")
sp.pprint(QTQ_symbol)

# Calculate determinant
det_symbol = sp.simplify(Q_symbol.det())

print("\ndet(Q):")
sp.pprint(det_symbol)

In [ ]:
# ============================================================
# NUMERICAL ORTHOGONALITY TEST
# ============================================================

QT = Q.T

# Matrix multiplication
QTQ = QT @ Q

# 2 × 2 identity matrix
I = np.eye(2)

print("Q:")
print(Q)

print("\nQ^T:")
print(QT)

print("\nQ^T Q:")
print(QTQ)

print("\nIdentity matrix I:")
print(I)

# Check whether Q^T Q = I
is_orthogonal = np.allclose(QTQ, I)

# Determinant
det_Q = np.linalg.det(Q)

print("\nIs Q orthogonal?", is_orthogonal)
print("det(Q) =", det_Q)

In [ ]:
# ============================================================
# APPLY ROTATION TO SCREEN COORDINATES
# ============================================================

# Mathematical transformation:
#
#        P' = QP

rotated_point = Q @ POINT

rotated_point_2 = Q @ POINT_2

print("Original Point 1 :", POINT)
print("Rotated Point 1  :", rotated_point)

print("\nOriginal Point 2 :", POINT_2)
print("Rotated Point 2  :", rotated_point_2)

In [ ]:
# ============================================================
# DISTANCE PRESERVATION
# ============================================================

# Euclidean distance between two points
original_distance = np.linalg.norm(POINT - POINT_2)

# Distance after rotation
rotated_distance = np.linalg.norm(
    rotated_point - rotated_point_2
)

print("Original distance :", original_distance)
print("Rotated distance  :", rotated_distance)

distance_preserved = np.isclose(
    original_distance,
    rotated_distance
)

print("\nDistance preserved?", distance_preserved)

In [ ]:
# ============================================================
# QUADRATIC FORM INVARIANCE
# ============================================================

# A point/vector P
P = POINT.reshape(2, 1)

# Original quadratic form:
# P^T P = x^2 + y^2
original_form = (P.T @ P)[0, 0]

# Rotated vector
QP = Q @ P

# Quadratic form after rotation
rotated_form = (QP.T @ QP)[0, 0]

print("Original P^T P       =", original_form)
print("After rotation       =", rotated_form)

print(
    "\nQuadratic form preserved?",
    np.isclose(original_form, rotated_form)
)

In [ ]:
# ============================================================
# ROTATING TWICE
# ============================================================

# First 90° rotation
P_after_90 = Q @ POINT

# Second 90° rotation
P_after_180 = Q @ P_after_90

# Matrix for two rotations
Q_squared = Q @ Q

# Theoretical 180° rotation matrix
Q_180 = np.array([
    [-1, 0],
    [0, -1]
])

print("Q²:")
print(Q_squared)

print("\nPoint after 90°:")
print(P_after_90)

print("\nPoint after 180°:")
print(P_after_180)

print("\nTheoretical 180° matrix:")
print(Q_180)

print(
    "\nDoes Q² represent a 180° rotation?",
    np.allclose(Q_squared, Q_180)
)

In [ ]:
# ============================================================
# PHONE SCREEN COORDINATES
# ============================================================

# ASSUMPTION:
# The origin (0,0) is at the CENTER of the screen.

half_width = SCREEN_WIDTH / 2
half_height = SCREEN_HEIGHT / 2

# Rectangle representing the portrait screen
screen = np.array([
    [-half_width, -half_height],
    [ half_width, -half_height],
    [ half_width,  half_height],
    [-half_width,  half_height],
    [-half_width, -half_height]
])

# Rotate every screen corner
rotated_screen = screen @ Q.T

print("Original screen coordinates:")
print(screen)

print("\nRotated screen coordinates:")
print(rotated_screen)

In [ ]:
# ============================================================
# CHART 1: PORTRAIT -> LANDSCAPE
# ============================================================

plt.figure(figsize=(10, 7))

# Original portrait screen
plt.plot(
    screen[:, 0],
    screen[:, 1],
    linewidth=3,
    label="Portrait screen"
)

# Rotated landscape screen
plt.plot(
    rotated_screen[:, 0],
    rotated_screen[:, 1],
    linewidth=3,
    label="Landscape after 90° rotation"
)

# Original point
plt.scatter(
    POINT[0],
    POINT[1],
    s=100,
    label="Original point"
)

# Rotated point
plt.scatter(
    rotated_point[0],
    rotated_point[1],
    s=100,
    label="Rotated point"
)

# Transformation arrow
plt.annotate(
    "",
    xy=rotated_point,
    xytext=POINT,
    arrowprops=dict(
        arrowstyle="->",
        linewidth=2
    )
)

# Coordinate axes
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

plt.title(
    "Phone Screen Rotation: Portrait → Landscape",
    fontsize=16,
    fontweight="bold"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.axis("equal")

plt.show()

In [ ]:
# ============================================================
# CHART 2: SUCCESSIVE ROTATIONS
# ============================================================

# 0° screen
screen_0 = screen

# 90° screen
screen_90 = screen @ Q.T

# 180° screen
screen_180 = screen @ Q_squared.T

plt.figure(figsize=(10, 7))

plt.plot(
    screen_0[:, 0],
    screen_0[:, 1],
    linewidth=3,
    label="0°"
)

plt.plot(
    screen_90[:, 0],
    screen_90[:, 1],
    linewidth=3,
    label="90°"
)

plt.plot(
    screen_180[:, 0],
    screen_180[:, 1],
    linewidth=3,
    label="180°"
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

plt.title(
    "Successive Screen Rotations",
    fontsize=16,
    fontweight="bold"
)

plt.legend()
plt.grid(True, alpha=0.3)
plt.axis("equal")

plt.show()

In [ ]:
# ============================================================
# BENCHMARK TEST DATASET
# ============================================================

# Small realistic set of screen-center coordinates.
# Coordinates are in pixels relative to the screen center.

benchmark_points = {
    "Center":   np.array([0.0, 0.0]),
    "Top":      np.array([0.0, 500.0]),
    "Right":    np.array([400.0, 0.0]),
    "Diagonal": np.array([300.0, 400.0]),
    "General":  np.array([-250.0, 350.0])
}

print("=" * 75)
print(
    f"{'Point':<12}"
    f"{'Original (x,y)':<25}"
    f"{'After 90°':<25}"
)
print("=" * 75)

for name, point in benchmark_points.items():

    transformed = Q @ point

    print(
        f"{name:<12}"
        f"{str(np.round(point, 2)):<25}"
        f"{str(np.round(transformed, 2)):<25}"
    )

In [ ]:
# ============================================================
# AUTOMATED TESTING
# ============================================================

print("=" * 75)
print("BENCHMARK VERIFICATION")
print("=" * 75)

all_tests_passed = True

for name, point in benchmark_points.items():

    # Rotate once
    rotated = Q @ point

    # -------------------------------
    # Test 1: Distance from origin
    # -------------------------------
    original_length = np.linalg.norm(point)
    rotated_length = np.linalg.norm(rotated)

    length_test = np.isclose(
        original_length,
        rotated_length
    )

    # -------------------------------
    # Test 2: Rotate twice
    # -------------------------------
    rotated_twice = Q @ rotated

    expected_180 = -point

    rotation_test = np.allclose(
        rotated_twice,
        expected_180
    )

    if not (length_test and rotation_test):
        all_tests_passed = False

    print(
        f"{name:<12}"
        f"Length preserved: {length_test}   "
        f"180° test: {rotation_test}"
    )

print("\nAll benchmark tests passed:", all_tests_passed)

In [ ]:
# ============================================================
# FINAL PROJECT REPORT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL MATHEMATICAL RESULTS")
print("=" * 70)

print("\n1. Rotation angle:")
print(f"   {ROTATION_ANGLE} degrees")

print("\n2. Rotation matrix Q:")
print(np.round(Q, 4))

print("\n3. Transpose Q^T:")
print(np.round(QT, 4))

print("\n4. Q^T Q:")
print(np.round(QTQ, 4))

print("\n5. Orthogonal?")
print("   ", is_orthogonal)

print("\n6. Determinant:")
print("   ", det_Q)

print("\n7. Original point:")
print("   ", POINT)

print("\n8. Point after 90° rotation:")
print("   ", np.round(rotated_point, 4))

print("\n9. Original distance:")
print("   ", round(original_distance, 4))

print("\n10. Rotated distance:")
print("    ", round(rotated_distance, 4))

print("\n11. Distance preserved?")
print("    ", distance_preserved)

print("\n12. Q²:")
print(np.round(Q_squared, 4))

print("\n13. Q² = 180° rotation?")
print("    ", np.allclose(Q_squared, Q_180))

print("\n" + "=" * 70)
print("PROJECT CONCLUSION")
print("=" * 70)

print("""
The phone screen rotation is successfully modeled using a
2D rotation matrix.

Q^T Q = I
Therefore Q is an orthogonal matrix.

det(Q) = +1
Therefore the transformation is a pure rotation.

The transformation preserves distances and lengths.

The quadratic form P^T P remains unchanged.

Two successive 90° rotations produce a 180° rotation:

Q² = [-1   0]
     [ 0  -1]

Thus, the mathematical model demonstrates how screen rotation
can be represented using orthogonal transformations.
""")

In [ ]:
# ============================================================
# OPTIONAL: TEST A DIFFERENT ROTATION ANGLE
# ============================================================

test_angle = float(input("Enter rotation angle in degrees: "))

test_theta = np.deg2rad(test_angle)

Q_test = np.array([
    [np.cos(test_theta), -np.sin(test_theta)],
    [np.sin(test_theta),  np.cos(test_theta)]
])

test_point = POINT
test_result = Q_test @ test_point

print("\nRotation matrix:")
print(np.round(Q_test, 4))

print("\nOriginal point:")
print(test_point)

print("\nTransformed point:")
print(np.round(test_result, 4))

print("\nQ^T Q:")
print(np.round(Q_test.T @ Q_test, 4))

print("\nOrthogonal:",
      np.allclose(Q_test.T @ Q_test, np.eye(2)))

print("Determinant:",
      round(np.linalg.det(Q_test), 4))